In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor

In [2]:
X = pd.read_csv('./data/X_train.csv')
y = pd.read_csv('./data/y_train.csv')
X_test = pd.read_csv('./data/X_test.csv')
y = y.fillna(0)

In [3]:
def recupColonnes():
    col = []
    for i in range(1000):
        col.append("holed_"+str(i+1))
    return col

In [4]:
def nettoyageDonnees(data):
    col = recupColonnes()
    partie2 = data[col]
    partie1 = data.drop(columns=col)
    return partie1, partie2

In [5]:
X_train, X_train_holeds = nettoyageDonnees(X)
X_test,X_test_holeds = nettoyageDonnees(X_test)

In [6]:
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [7]:
def fill_nan_with_interpolation_linear(column):
    col = column.copy()
    col = col.interpolate(method='linear', limit_direction='both')
    return col

In [8]:
y_train_pred = X_train_holeds.apply(fill_nan_with_interpolation_linear,axis=0)
y_train_copy = y.copy()
y_train_copy = y_train_copy.drop(columns=["Horodate"])
mae_train = mean_absolute_error(y_train_copy.to_numpy().flatten(), y_train_pred.to_numpy().flatten())
print("MAE Train:", mae_train)

MAE Train: 12.428300483575784


In [9]:
def fill_nan_with_interpolation_spline(column):
    col = column.copy()
    if col.notna().sum()<2:
        return col.fillna(0)
    if col.notna().sum()<4:
        return col.interpolate(method='linear', limit_direction='both')
    col = col.interpolate(method='spline', limit_direction='both',order=3)
    return col

In [10]:
y_train_pred = X_train_holeds.apply(fill_nan_with_interpolation_spline,axis=0)
y_train_copy = y.copy()
y_train_copy = y_train_copy.drop(columns=["Horodate"])
mae_train = mean_absolute_error(y_train_copy.to_numpy().flatten(), y_train_pred.to_numpy().flatten())
print("MAE Train:", mae_train)

c:\Users\Lenovo\anaconda3\Lib\site-packages\pandas\core\missing.py:604: UserWarning: 
The maximal number of iterations maxit (set to 20 by the program)
allowed for finding a smoothing spline with fp=s has been reached: s
too small.
There is an approximation returned but the corresponding weighted sum
of squared residuals does not satisfy the condition abs(fp-s)/s < tol.
  terp = interpolate.UnivariateSpline(x, y, k=order, **kwargs)


MAE Train: 151950.19403430136


In [11]:
def knn_impute(df, n_neighbors=5):
    imputer = KNNImputer(n_neighbors=n_neighbors,weights="distance")
    col_imputed = imputer.fit_transform(df)
    return pd.DataFrame(col_imputed, columns=df.columns, index=df.index)

In [12]:
y_train_pred = knn_impute(X_train_holeds)
y_train_copy = y.copy()
y_train_copy = y_train_copy.drop(columns=["Horodate"])
mae_train = mean_absolute_error(y_train_copy.to_numpy().flatten(), y_train_pred.to_numpy().flatten())
print("MAE Train:", mae_train)

MAE Train: 10.214014009599783


In [137]:
y_test_pred = knn_impute(X_test_holeds)
dates = X_test["Horodate"]
y_test_pred.insert(0, "Horodate", dates)

In [139]:
y_test_pred.to_csv('./data/y_test.csv', index=False)

In [17]:
def forest_regression_version(df_users,df_holed):
    df_users = df_users.copy()
    df_users = df_users.iloc[:,1:]
    df_holed = df_holed.copy()
    df_users.columns = df_users.columns.str.strip()
    df_holed.columns = df_holed.columns.str.strip()
    result = df_holed.copy()
    user_cols = df_users.columns
    for col in df_holed.columns:
        train_mask = df_holed[col].notna()
        test_mask = df_holed[col].isna()
        if test_mask.sum() == 0:
            continue
        X_train = df_users.loc[train_mask, user_cols]
        y_train = df_holed.loc[train_mask, col]
        X_test = df_users.loc[test_mask, user_cols]
        model = RandomForestRegressor(n_estimators=2,random_state=42)
        model.fit(X_train, y_train)
        result.loc[test_mask, col] = model.predict(X_test)
    return result

In [ ]:
y_train_pred = forest_regression_version(X_train, X_train_holeds)